# Day 052 Project: Ship an AI API

## What You're Building

A complete, runnable **FastAPI service** backed by local Ollama. You run one command — `uvicorn main:app --reload` — and get a live HTTP API with `/health`, `/templates`, `/chat`, and `/render/{name}`, plus automatic interactive docs at `/docs`. That `main.py` is the deliverable.

## Project Requirements

1. Use the provided `build_api` factory (built across Exercises 1–5).
2. Build the app and exercise every route with `TestClient` to prove it works in-process.
3. Call `write_api_app('main.py')` to generate the real service file.
4. Run `_run_project_checks()` to verify the file is well-formed.
5. Then, in a terminal: `uvicorn main:app --reload` and open `http://localhost:8000/docs`.

## Bonus Challenges

- Add a `GET /models` route that returns `ollama.list()` model names.
- Add a `max_tokens` field to `ChatRequest` and pass it through `options={'num_predict': ...}`.
- Add CORS with `from fastapi.middleware.cors import CORSMiddleware` so a browser front-end (tomorrow, Day 53) can call it.

## Provided: All Logic (Exercises 1–5)

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import ollama


class ChatRequest(BaseModel):
    """Request body for the chat endpoints."""
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    """Response body returned by the chat endpoints."""
    reply: str
    model: str


class HealthResponse(BaseModel):
    """Response body for the health check."""
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    """Call Ollama once and return the reply text. Raises on model error."""
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    """Assemble the complete AI API: health, templates, chat, and templated chat."""
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app

## Provided: main.py Builder

In [ ]:
from pathlib import Path

# The full FastAPI app source (models + templates + run_model + build_api +
# a uvicorn entry point). Embedded as a string so we can write it to a real
# file — the runnable deliverable you launch with `uvicorn main:app`.
_MAIN_SRC = 'import warnings\nwarnings.filterwarnings(\'ignore\')\nfrom fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel, Field\nimport ollama\n\n\nclass ChatRequest(BaseModel):\n    """Request body for the chat endpoints."""\n    message: str = Field(min_length=1, description=\'User message for the model\')\n    temperature: float = Field(default=0.7, ge=0.0, le=1.0)\n\n\nclass ChatResponse(BaseModel):\n    """Response body returned by the chat endpoints."""\n    reply: str\n    model: str\n\n\nclass HealthResponse(BaseModel):\n    """Response body for the health check."""\n    status: str\n    model: str\n\n\nPROMPT_TEMPLATES = {\n    \'summary\':  \'Summarize the following topic in two sentences: {topic}\',\n    \'explain\':  \'Explain {topic} to a complete beginner.\',\n    \'critique\': \'List three criticisms of {topic}.\',\n}\n\n\ndef run_model(model: str, prompt: str, temperature: float = 0.7) -> str:\n    """Call Ollama once and return the reply text. Raises on model error."""\n    resp = ollama.chat(\n        model=model,\n        messages=[{\'role\': \'user\', \'content\': prompt}],\n        options={\'temperature\': temperature},\n    )\n    return resp[\'message\'][\'content\'].strip()\n\n\ndef build_api(model: str = \'llama3.2\') -> FastAPI:\n    """Assemble the complete AI API: health, templates, chat, and templated chat."""\n    app = FastAPI(title=\'AI API\', version=\'1.0.0\')\n\n    @app.get(\'/health\', response_model=HealthResponse)\n    def health():\n        return HealthResponse(status=\'ok\', model=model)\n\n    @app.get(\'/templates\')\n    def list_templates():\n        return {\'templates\': list(PROMPT_TEMPLATES.keys())}\n\n    @app.post(\'/chat\', response_model=ChatResponse)\n    def chat(req: ChatRequest):\n        try:\n            return ChatResponse(reply=run_model(model, req.message, req.temperature),\n                                model=model)\n        except Exception as e:\n            raise HTTPException(status_code=503, detail=f\'Model unavailable: {e}\')\n\n    @app.post(\'/render/{name}\', response_model=ChatResponse)\n    def render_chat(name: str, req: ChatRequest):\n        if name not in PROMPT_TEMPLATES:\n            raise HTTPException(status_code=404, detail=f\'template {name!r} not found\')\n        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)\n        try:\n            return ChatResponse(reply=run_model(model, prompt, req.temperature),\n                                model=model)\n        except Exception as e:\n            raise HTTPException(status_code=503, detail=f\'Model unavailable: {e}\')\n\n    return app\n\n\napp = build_api()\n\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(app, host=\'0.0.0.0\', port=8000)\n'


def write_api_app(path: str = 'main.py') -> str:
    """Write the self-contained FastAPI app to `path` and return the path."""
    Path(path).write_text(_MAIN_SRC, encoding='utf-8')
    return path

## Your Pipeline

In [ ]:
# TODO: app = build_api()
# TODO: client = TestClient(app)
# TODO: print('health :', client.get('/health').json())
# TODO: print('chat   :', client.post('/chat', json={'message': 'Say hello in 3 words.'}).json())
# TODO: print('render :', client.post('/render/summary', json={'message': 'FastAPI'}).json())
#
# TODO: path = write_api_app('main.py')
# TODO: print('Wrote', path)
# TODO: print('Run it with:  uvicorn main:app --reload')

## Checks

In [ ]:
import os


def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: app built and served in-process
    try:
        assert 'app' in globals() and isinstance(app, FastAPI), 'create app = build_api()'
        client = TestClient(app)
        assert client.get('/health').status_code == 200
        passed += 1; print('✅ Check 1: build_api() app responds to /health')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: main.py written
    try:
        assert os.path.exists('main.py'), 'main.py not found — call write_api_app()'
        passed += 1; print('✅ Check 2: main.py exists')
    except Exception as e:
        print(f'❌ Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    src = open('main.py', encoding='utf-8').read()

    # Check 3: main.py imports FastAPI and defines build_api
    try:
        assert 'from fastapi import FastAPI' in src, 'main.py must import FastAPI'
        assert 'def build_api' in src, 'main.py must define build_api'
        assert 'app = build_api()' in src, 'main.py must expose module-level app'
        passed += 1; print('✅ Check 3: main.py has FastAPI + build_api + app')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: main.py has a uvicorn entry point
    try:
        assert 'uvicorn' in src, 'main.py should include a uvicorn run guard'
        assert "__main__" in src, 'main.py should guard the run under __main__'
        passed += 1; print('✅ Check 4: main.py has a uvicorn entry point')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: main.py is valid Python (compiles)
    try:
        compile(src, 'main.py', 'exec')
        passed += 1; print('✅ Check 5: main.py compiles as valid Python')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Project complete! Run: uvicorn main:app --reload')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()